# Q1 재무 변화 분석 — 안지형 담당

Issue #6. 실제 Open DART SQLite `finance`를 Pandas로 분석하고 강동윤 SQL 결과와 같은 키로 대조한다. 검증 전에는 위험·원인을 단정하지 않는다.

In [1]:
from pathlib import Path
from contextlib import closing
import sqlite3, sys
import pandas as pd
root = Path.cwd()
if not (root / 'src').exists():
    root = root.parent
sys.path.insert(0, str(root))
from src.finance_pandas_anjihyeong import analyze, KEYS
from src.validate_results import compare
database = root / 'data' / 'processed' / 'dart.sqlite'
assert database.is_file(), f'먼저 로컬 DART DB를 준비하세요: {database}'
print('Pandas:', pd.__version__)

C:\Users\USER\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Pandas: 3.0.4


## 원본 로드와 독립 계산

사업보고서(11011)·원화·회사·연도·CFS/OFS를 구분한다. 중복 계정은 `ord`가 가장 작은 원본 한 행을 대표로 사용하고 `source_rows`로 중복 수를 남긴다.

In [2]:
with closing(sqlite3.connect(database)) as db:
    raw = pd.read_sql_query('SELECT * FROM finance', db)
    db.executescript((root / 'sql' / 'queries_finance.sql').read_text(encoding='utf-8'))
    sql_result = pd.read_sql_query('SELECT * FROM finance_q1_changes', db)
pandas_result = analyze(raw)
print('원본 재무 행:', len(raw), 'Pandas 선택 지표 행:', len(pandas_result), '중복 원본 그룹:', int(pandas_result.source_rows.gt(1).sum()))

원본 재무 행: 60 Pandas 선택 지표 행: 16 중복 원본 그룹: 4


In [3]:
sample = pandas_result.loc[
    pandas_result.corp_code.eq('00126380') &
    pandas_result.bsns_year.eq('2024') &
    pandas_result.fs_div.eq('CFS')
]
display(sample[['account_nm', 'current_amount', 'previous_amount', 'change_amount', 'change_pct', 'source_rows']])

,account_nm,current_amount,previous_amount,change_amount,change_pct,source_rows
8,자산총계,514531948000000,455905980000000,58625968000000,12.859223,1
9,당기순이익(손실),34451351000000,15487100000000,18964251000000,122.451918,2
10,매출액,300870903000000,258935494000000,41935409000000,16.195311,1
11,영업이익,32725961000000,6566976000000,26158985000000,398.341413,1


## SQL 교차검증

회사·사업연도·보고서·CFS/OFS·재무제표 유형·계정 키를 맞춰 금액, 증감액, 증감률, 원본 중복 수를 비교한다. 전기 0의 증감률은 비워 둔다.

In [4]:
report = compare(sql_result, pandas_result, KEYS,
                 ['source_rows', 'current_amount', 'previous_amount',
                  'change_amount', 'change_pct'], tolerance=1e-6)
assert report['status'] == 'PASS', report
print('SQL-Pandas 일치:', report['rows'], '행 / 불일치:', report['mismatch_count'], '건')

SQL-Pandas 일치: 16 행 / 불일치: 0 건


## 해석 범위

위 출력은 같은 로컬 Open DART 원본의 두 계산 결과가 일치하는지 보여준다. 원본 자체의 오류, 공시 패턴(Q2), 이상징후 원인은 별도 확인이 필요하다. 2024년 전기 금액을 2023년 당기와 별개로 합산하지 않는다.